# [16.8] Do SHAPley and Mechanistic Interpretability Agree?

This notebook asks one bounded question: when can SHAPley-style attribution and a mechanistic ground-truth score be said to agree?

You will not answer this with a pretty heatmap. You will answer it with finite games, exact tests, deletion/insertion consequences, and a CUDA model organism with a shuffled-label control.

The key lesson is that disagreement is often about the **player set**. A single-feature Shapley value can miss an XOR mechanism because the real player is a pair.

## Core Question

When attribution and mechanistic scores disagree, are the scores wrong, or are they scoring different player sets?


## Learning Objectives

By the end, you should be able to:

1. Rank attribution scores deterministically.
2. Convert a ranking into deletion and insertion consequence curves.
3. Check additive agreement against a known mechanistic score.
4. Diagnose an XOR disagreement where interactions are the correct player set.
5. Write an agreement matrix and visible artifact bundle.
6. Interpret the CUDA report without making broad large-model claims.

<details>
<summary>Help - what counts as agreement?</summary>

Agreement means the top players and consequences line up: high rank correlation, exact top-k overlap, top deletion beats a non-top baseline, and the correct interaction pair is recovered.

</details>


In [1]:
import csv
import json
import sys
from pathlib import Path

import torch as t

chapter = "chapter16_shapley_attribution_baselines"
section = "part8_shapley_mechinterp_agreement"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part8_shapley_mechinterp_agreement.tests as tests
import part8_shapley_mechinterp_agreement.utils as utils

from arena_ext.shapley_attribution import (
    additive_game,
    attribution_agreement_report,
    interaction_agreement_report,
    pairwise_shapley_interactions,
    topk_overlap_fraction,
    xor_game,
)
from arena_ext.shapley_neural_game import (
    NEURAL_GAME_NUM_PLAYERS,
    coalition_table_from_true_game,
)

MAIN = True
ARTIFACT_DIR = section_dir / "artifacts"


## Exercise 1 - rank attribution scores

> ```yaml
> Difficulty: medium
> Importance: high
> ```


Implement a deterministic descending rank helper.

In [2]:
def _rank_desc(scores: t.Tensor) -> list[int]:
    """Return indices sorted by descending score."""
    return [int(item) for item in t.argsort(scores.detach().double().cpu(), descending=True)]


if MAIN:
    tests.test_rank_desc_toy_oracle(_rank_desc)


All tests in `test_rank_desc_toy_oracle` passed!


<details>
<summary>Expected output</summary>

`test_rank_desc_toy_oracle` should pass. The tie case keeps a deterministic order.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 2 - analytic mechanism scores

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [3]:
def analytic_neural_game_mechanistic_scores() -> t.Tensor:
    """Return feature scores from the generated rule decomposition."""
    return t.tensor([2.3, -1.45, 2.7, 0.15], dtype=t.float64)


if MAIN:
    tests.test_analytic_neural_game_mechanistic_scores_toy_oracle(
        analytic_neural_game_mechanistic_scores
    )


All tests in `test_analytic_neural_game_mechanistic_scores_toy_oracle` passed!


<details>
<summary>Expected output</summary>

The analytic scores should be `[2.3, -1.45, 2.7, 0.15]`.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 3 - consequence curves

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [4]:
def _curve_from_rank(values: dict[frozenset[int], float], rank: list[int], mode: str) -> list[dict]:
    """Build deletion or insertion curve points from a ranking."""
    if mode not in {"deletion", "insertion"}:
        raise ValueError("mode must be 'deletion' or 'insertion'.")

    active: set[int] = set(range(NEURAL_GAME_NUM_PLAYERS)) if mode == "deletion" else set()
    points = [{"step": 0, "player": "start", "value": values[frozenset(active)]}]
    for step, player in enumerate(rank, start=1):
        if mode == "deletion":
            active.remove(player)
        else:
            active.add(player)
        points.append({"step": step, "player": player, "value": values[frozenset(active)]})
    return points


if MAIN:
    tests.test_curve_from_rank_deletion_and_insertion(_curve_from_rank)


All tests in `test_curve_from_rank_deletion_and_insertion` passed!


<details>
<summary>Expected output</summary>

Deletion starts from the full coalition; insertion starts from the empty coalition.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 4 - additive agreement positive control

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [5]:
def _tensor_report(report) -> dict:
    result = report.__dict__.copy()
    for key, value in list(result.items()):
        if hasattr(value, "tolist"):
            result[key] = value.tolist()
    return result


def additive_agreement_smoke_test() -> dict:
    """Return rank, top-k, and deletion metrics for the additive positive control."""
    mechanistic_scores = t.tensor([1.0, 2.0, 0.5])
    values = additive_game(mechanistic_scores)
    return _tensor_report(
        attribution_agreement_report(
            values,
            mechanistic_scores=mechanistic_scores,
            num_players=3,
        )
    )


if MAIN:
    tests.test_additive_agreement_smoke_test(additive_agreement_smoke_test)


All tests in `test_additive_agreement_smoke_test` passed!


<details>
<summary>Expected output</summary>

`topk_overlap == 1.0`, `spearman_correlation > 0.99`, and top deletion beats baseline.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 5 - XOR disagreement control

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [6]:
def xor_disagreement_smoke_test() -> dict:
    """Return ordinary Shapley and pair-interaction metrics for the XOR control."""
    return _tensor_report(interaction_agreement_report(xor_game()))


if MAIN:
    tests.test_xor_disagreement_smoke_test(xor_disagreement_smoke_test)


All tests in `test_xor_disagreement_smoke_test` passed!


<details>
<summary>Expected output</summary>

Ordinary Shapley has zero single-feature value, while pair interaction recovers value `2.0`.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 6 - agreement artifact bundle

> ```yaml
> Difficulty: hard
> Importance: high
> ```


In [7]:
def _artifact_display_path(path: Path) -> str:
    try:
        return str(path.relative_to(root_dir))
    except ValueError:
        return str(path)


def _write_curve_plot(path: Path, title: str, ylabel: str, curves: dict[str, list[dict]]) -> None:
    import matplotlib

    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=160)
    colors = {
        "trained_shapley_rank": "#1f77b4",
        "mechanistic_rank": "#2ca02c",
        "shuffled_control_rank": "#d62728",
    }
    markers = {
        "trained_shapley_rank": "o",
        "mechanistic_rank": "s",
        "shuffled_control_rank": "^",
    }
    linestyles = {
        "trained_shapley_rank": "-",
        "mechanistic_rank": "--",
        "shuffled_control_rank": "-",
    }
    for label in ("mechanistic_rank", "trained_shapley_rank", "shuffled_control_rank"):
        points = curves[label]
        xs = [point["step"] for point in points]
        ys = [point["value"] for point in points]
        ax.plot(
            xs,
            ys,
            marker=markers[label],
            linestyle=linestyles[label],
            linewidth=2.3,
            markersize=5,
            color=colors[label],
            label=label,
            markerfacecolor="white" if label == "trained_shapley_rank" else colors[label],
            markeredgewidth=1.8,
            zorder=3 if label == "trained_shapley_rank" else 2,
        )
    ax.set_title(title)
    ax.set_xlabel("players removed" if "Deletion" in title else "players inserted")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(NEURAL_GAME_NUM_PLAYERS + 1))
    ax.grid(True, alpha=0.28)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def _write_heatmap(path: Path, rows: list[str], columns: list[str], values: list[list[float]]) -> None:
    import matplotlib

    matplotlib.use("Agg", force=True)
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(7.5, 3.7), dpi=160)
    image = ax.imshow(values, cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(columns)), labels=columns)
    ax.set_yticks(range(len(rows)), labels=rows)
    ax.set_title("Top-k overlap with analytic mechanism")
    for row_idx, row_values in enumerate(values):
        for col_idx, value in enumerate(row_values):
            text_color = "white" if value < 0.55 else "black"
            ax.text(col_idx, row_idx, f"{value:.2f}", ha="center", va="center", color=text_color)
    fig.colorbar(image, ax=ax, shrink=0.85, label="overlap")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def write_agreement_artifacts(
    *,
    output_dir: Path = ARTIFACT_DIR,
    model_values: dict[frozenset[int], float],
    true_values: dict[frozenset[int], float],
    shuffled_values: dict[frozenset[int], float],
    agreement,
    shuffled_agreement,
    model_interactions: t.Tensor,
    true_interactions: t.Tensor,
) -> dict:
    """Write the roadmap-required agreement matrix and consequence plots."""

    output_dir.mkdir(parents=True, exist_ok=True)

    interaction_error = float((model_interactions - true_interactions).abs().max().item())
    additive = additive_agreement_smoke_test()
    xor = xor_disagreement_smoke_test()
    matrix_rows = [
        {
            "task": "additive_control",
            "method_a": "ExactShapley",
            "method_b": "MechanisticScores",
            "player_type_a": "feature",
            "player_type_b": "feature",
            "metric": "spearman_rank_correlation",
            "value": f"{additive['spearman_correlation']:.6g}",
            "interpretation": "Additive ground truth gives full rank agreement.",
        },
        {
            "task": "neural_coalition_game",
            "method_a": "ExactShapley",
            "method_b": "AnalyticMechanisticScores",
            "player_type_a": "feature",
            "player_type_b": "feature",
            "metric": "spearman_rank_correlation",
            "value": f"{agreement.spearman_correlation:.6g}",
            "interpretation": "Trained model ablations recover the analytic feature ordering.",
        },
        {
            "task": "neural_coalition_game",
            "method_a": "ExactShapley",
            "method_b": "AnalyticMechanisticScores",
            "player_type_a": "feature",
            "player_type_b": "feature",
            "metric": "top2_overlap",
            "value": f"{agreement.topk_overlap:.6g}",
            "interpretation": "Top causal features match the analytic mechanism.",
        },
        {
            "task": "neural_coalition_game",
            "method_a": "ExactShapley",
            "method_b": "FeatureDeletion",
            "player_type_a": "feature",
            "player_type_b": "behavior",
            "metric": "deletion_drop_minus_baseline",
            "value": f"{agreement.deletion_drop - agreement.random_baseline_drop:.6g}",
            "interpretation": "Deleting the top Shapley feature hurts more than deleting a non-top baseline.",
        },
        {
            "task": "neural_coalition_game",
            "method_a": "ShapleyInteractions",
            "method_b": "AnalyticPairInteractions",
            "player_type_a": "feature_pair",
            "player_type_b": "feature_pair",
            "metric": "max_abs_error",
            "value": f"{interaction_error:.6g}",
            "interpretation": "Pair interactions recover the planted positive and negative feature pairs.",
        },
        {
            "task": "shuffled_label_control",
            "method_a": "ExactShapley",
            "method_b": "AnalyticMechanisticScores",
            "player_type_a": "feature",
            "player_type_b": "feature",
            "metric": "spearman_rank_correlation",
            "value": f"{shuffled_agreement.spearman_correlation:.6g}",
            "interpretation": "The shuffled-label trained model fails the mechanistic agreement test.",
        },
        {
            "task": "xor_control",
            "method_a": "OrdinaryShapley",
            "method_b": "ShapleyInteractions",
            "player_type_a": "feature",
            "player_type_b": "feature_pair",
            "metric": "disagreement_detected",
            "value": "1",
            "interpretation": "Single-feature Shapley misses XOR while pair interactions recover it.",
        },
    ]
    matrix_path = output_dir / "agreement_matrix.csv"
    with matrix_path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(matrix_rows[0]), lineterminator="\n")
        writer.writeheader()
        writer.writerows(matrix_rows)

    shapley_rank = _rank_desc(agreement.shapley_values)
    mechanistic_rank = _rank_desc(agreement.mechanistic_scores)
    shuffled_rank = _rank_desc(shuffled_agreement.shapley_values)
    deletion_curves = {
        "trained_shapley_rank": _curve_from_rank(model_values, shapley_rank, "deletion"),
        "mechanistic_rank": _curve_from_rank(model_values, mechanistic_rank, "deletion"),
        "shuffled_control_rank": _curve_from_rank(model_values, shuffled_rank, "deletion"),
    }
    insertion_curves = {
        "trained_shapley_rank": _curve_from_rank(model_values, shapley_rank, "insertion"),
        "mechanistic_rank": _curve_from_rank(model_values, mechanistic_rank, "insertion"),
        "shuffled_control_rank": _curve_from_rank(model_values, shuffled_rank, "insertion"),
    }
    _write_curve_plot(
        output_dir / "deletion_curves.png",
        "Deletion Consequences",
        "model value after deletion",
        deletion_curves,
    )
    _write_curve_plot(
        output_dir / "insertion_curves.png",
        "Insertion Consequences",
        "model value after insertion",
        insertion_curves,
    )

    columns = [f"k={k}" for k in range(1, NEURAL_GAME_NUM_PLAYERS + 1)]
    heatmap_rows = [
        "trained_shapley",
        "trained_patching",
        "shuffled_shapley",
        "shuffled_patching",
    ]
    heatmap_values = [
        [
            topk_overlap_fraction(agreement.shapley_values, agreement.mechanistic_scores, k=k)
            for k in range(1, NEURAL_GAME_NUM_PLAYERS + 1)
        ],
        [
            topk_overlap_fraction(agreement.patching_effects, agreement.mechanistic_scores, k=k)
            for k in range(1, NEURAL_GAME_NUM_PLAYERS + 1)
        ],
        [
            topk_overlap_fraction(shuffled_agreement.shapley_values, agreement.mechanistic_scores, k=k)
            for k in range(1, NEURAL_GAME_NUM_PLAYERS + 1)
        ],
        [
            topk_overlap_fraction(
                shuffled_agreement.patching_effects,
                agreement.mechanistic_scores,
                k=k,
            )
            for k in range(1, NEURAL_GAME_NUM_PLAYERS + 1)
        ],
    ]
    _write_heatmap(output_dir / "topk_overlap_heatmap.png", heatmap_rows, columns, heatmap_values)

    disagreement_path = output_dir / "method_disagreement_examples.md"
    disagreement_path.write_text(
        "\n".join(
            [
                "# Method Disagreement Examples",
                "",
                "## Agreement case: additive and trained finite game",
                "",
                (
                    "Exact Shapley and analytic mechanistic scores agree on the trained "
                    f"neural coalition game: Spearman={agreement.spearman_correlation:.3f}, "
                    f"top-2 overlap={agreement.topk_overlap:.3f}."
                ),
                (
                    "Deleting the top Shapley feature drops the model value by "
                    f"{agreement.deletion_drop:.3f}, above the non-top baseline "
                    f"{agreement.random_baseline_drop:.3f}."
                ),
                "",
                "## Disagreement case: XOR interaction",
                "",
                (
                    "Ordinary single-feature Shapley has max absolute value "
                    f"{xor['max_single_feature_value']:.3f} on XOR, so it misses the "
                    "mechanism when players are individual features."
                ),
                (
                    "Pairwise Shapley interaction recovers the causal pair with value "
                    f"{xor['recovered_pair_interaction']:.3f}. This is a tested "
                    "player-set disagreement, not a visual story."
                ),
                "",
                "## Negative control: shuffled trained model",
                "",
                (
                    "The shuffled-label control fits its own targets but fails agreement: "
                    f"Spearman={shuffled_agreement.spearman_correlation:.3f}, "
                    f"top-2 overlap={shuffled_agreement.topk_overlap:.3f}."
                ),
                "",
            ]
        )
    )

    paths = [
        matrix_path,
        output_dir / "deletion_curves.png",
        output_dir / "insertion_curves.png",
        output_dir / "topk_overlap_heatmap.png",
        disagreement_path,
    ]
    return {
        "agreement_artifacts_written": all(path.exists() and path.stat().st_size > 0 for path in paths),
        "agreement_artifact_count": len(paths),
        "agreement_matrix_rows": len(matrix_rows),
        "agreement_case_count": 2,
        "disagreement_case_count": 1,
        "deletion_curve_points": len(next(iter(deletion_curves.values()))),
        "insertion_curve_points": len(next(iter(insertion_curves.values()))),
        "topk_heatmap_rows": len(heatmap_rows),
        "topk_heatmap_cols": len(columns),
        "agreement_artifact_paths": [_artifact_display_path(path) for path in paths],
    }


if MAIN:
    tests.test_write_agreement_artifacts_contract()


All tests in `test_write_agreement_artifacts_contract` passed!


<details>
<summary>Expected output</summary>

The artifact writer creates a seven-row matrix, two curve PNGs, a heatmap PNG, and a Markdown disagreement note.

</details>

<details>
<summary>Solution</summary>

The implementation is in the code cell above.

</details>


## Exercise 7 - notebook contract

> ```yaml
> Difficulty: medium
> Importance: medium
> ```


In [8]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    return {
        "additive_agreement": additive_agreement_smoke_test(),
        "xor_disagreement": xor_disagreement_smoke_test(),
    }


if MAIN:
    tests.test_notebook_contract(run_smoke_test)


All tests in `test_notebook_contract` passed!


## Exercise 8 - inspect committed CUDA report

> ```yaml
> Difficulty: medium
> Importance: high
> ```


In [9]:
def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    """Run the real CUDA finite-game agreement preflight for this section."""
    from chapter16_shapley_attribution_baselines.exercises.part8_shapley_mechinterp_agreement import solutions as reference_solutions

    return reference_solutions.run_gpu_test(max_vram_gb=max_vram_gb)


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    """Run the validated experiment path used by verification_report.json."""
    return run_gpu_test(max_vram_gb=max_vram_gb)


In [10]:
report_path = section_dir / "verification_report.json"
report = json.loads(report_path.read_text())
gpu = report["metrics"]["gpu_test"]

summary = {
    "preflight_passed": gpu["preflight_passed"],
    "model_family": gpu["model_family"],
    "spearman_correlation": gpu["spearman_correlation"],
    "topk_overlap": gpu["topk_overlap"],
    "deletion_gap": gpu["deletion_drop"] - gpu["random_baseline_drop"],
    "interaction_max_abs_error": gpu["interaction_max_abs_error"],
    "shuffled_control_rejected": gpu["shuffled_control_rejected"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}
utils.print_report("Committed CUDA agreement report", summary)

if MAIN:
    tests.test_committed_gpu_report_records_agreement_and_controls()



Committed CUDA agreement report
  preflight_passed: True
  model_family: cuda_trained_neural_coalition_game_mlp
  spearman_correlation: 0.9999999999999998
  topk_overlap: 1.0
  deletion_gap: 3.6000013103087745
  interaction_max_abs_error: 4.52001889694742e-07
  shuffled_control_rejected: True
  peak_vram_gb: 0.06262922286987305
All tests in `test_committed_gpu_report_records_agreement_and_controls` passed!


## Signature Result

The committed CUDA report should show:

| Case | Metric | Expected |
| --- | ---: | ---: |
| trained finite game | Spearman | about 1.0 |
| trained finite game | top-k overlap | 1.0 |
| top deletion | beats non-top baseline | true |
| interaction recovery | max error | less than 1e-4 |
| shuffled control | rejected | true |

<details>
<summary>Interpreting the result</summary>

The positive result is bounded: exact model-output Shapley agrees with analytic mechanism scores on a finite generated rule.
The negative control matters because a trained model can fit shuffled targets while failing the mechanism agreement test.

</details>


## Limitations and Bonus Anomaly Hunting

This notebook does not prove method agreement on arbitrary transformer circuits. It proves a finite protocol: define the player set, test consequences, recover interactions, and reject shuffled controls.

Bonus ideas:

1. Add a three-way interaction where pair interactions fail.
2. Swap the player set from features to edges and see which agreement rows survive.
3. Repeat the CUDA finite game across seeds and look for spurious shuffled alignment.
